# Bonus: SQL Agent

Give an agent a SQL tool over the Chinook sample database and let it discover the schema through trial and error.

**Key learnings**
- Expose SQL as a tool over a sample database
- Agents iterate on errors to learn the schema
- Always inspect the final query the agent ran



### Load environment variables

`load_dotenv()` initializes the environment by loading the `.env` file so the model provider API key is available.

**Key learnings**
- Expose SQL as a tool over a sample database
- Agents iterate on errors to learn the schema
- Always inspect the final query the agent ran


In [ ]:
from dotenv import load_dotenv

load_dotenv()

### Connect to the SQL database

`SQLDatabase.from_uri` opens the local `Chinook.db` SQLite database (a sample music store schema with artists, albums, tracks, etc.). This handle lets us run SQL against it.

In [ ]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

### Define a SQL query tool

`sql_query` lets an agent run arbitrary SQL and returns the result, catching errors and returning them as text so the agent can read the message and correct its query. The final line does a quick sanity check by querying the first 10 artists.

In [ ]:
from langchain.tools import tool

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

sql_query.invoke("SELECT * FROM Artist LIMIT 10")

### Create the SQL agent

`create_agent` builds an agent with the `sql_query` tool. With no system prompt, the model relies on the tool description and the errors it gets back to explore the schema and refine its queries.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[sql_query]
)

### Ask a question

We ask for the most popular artist starting with 'S'. The agent will iteratively write SQL, learn the real schema from the errors it hits, and converge on a working query.

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")

response = agent.invoke(
    {"messages": [question]}
)

### Inspect the full message history

`pprint(response['messages'])` reveals the agent's trial-and-error: it guesses table/column names, gets errors, inspects the schema with `sqlite_master` and `PRAGMA`, and finally joins Artist/Album/Track to count tracks and answer the question.

In [ ]:
from pprint import pprint

pprint(response['messages'])

### Show the final SQL query

This prints the SQL from the agent's last successful tool call, the query that actually produced the answer after the earlier attempts failed.

In [ ]:
print(response["messages"][-3].tool_calls[0]['args']['query'])